# ECDC DATA FROM 2018 TO 2023

In [ ]:
!pip install pycountry

In [ ]:
from io import BytesIO, StringIO
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import pycountry

file_urls = {
    2018: "https://www.ecdc.europa.eu/sites/default/files/documents/End-of-the%20season%20update%202018_table.xlsx",
    2019: "https://www.ecdc.europa.eu/sites/default/files/documents/WNV-end-of-seaon-update.xlsx",
    2020: "https://www.ecdc.europa.eu/sites/default/files/documents/west-nile-virus-july-to%20december-2020-transmission-season.xlsx",
    2021: "https://www.ecdc.europa.eu/sites/default/files/documents/WNV_June_to_Nov_2021_0.xlsx",
    2022: "https://www.ecdc.europa.eu/sites/default/files/documents/WNV_2022.csv",
    2023: "https://www.ecdc.europa.eu/sites/default/files/documents/transmission-WNV-2023-season.xlsx",
}

column_mapping = {
    2018: {
        "Country": "country",
        "Region (NUTS 3 level)": "location_name",
        "First human case reported": "first_case_date",
        "Number of human cases": "human_cases",
        "Number of confirmed human cases": "confirmed_human_cases"
    },
    2019: {
        "Country": "country",
        "Region (NUTS 3/ GAUL 1 level)": "location_name",
        "First human case reported": "first_case_date",
        "Number of human cases": "human_cases",
        "Number of confirmed human cases": "confirmed_human_cases"
    },
    2020: {
        "Country": "country",
        "Region (NUTS 3/ GAUL 1 level)": "location_name",
        "First human case reported": "first_case_date",
        "Number of human cases": "human_cases",
        "Number of confirmed human cases": "confirmed_human_cases"
    },
    2021: {
        "Place Of Infection": "location_code",
        "Location Name": "location_name",
        "First Human case reported": "first_case_date",
        "Number of human cases": "human_cases",
        "Number of confirmed human cases": "confirmed_human_cases",
    },
    2022: {
        "Place Of Infection": "location_code",
        "Location Name": "location_name",
        "First Human case reported": "first_case_date",
        "Number of human cases": "human_cases",
        "Number of confirmed human cases": "confirmed_human_cases"
    },
    2023: {
        "Place Of Infection": "location_code",
        "Location Name": "location_name",
        "First Human case reported": "first_case_date",
        "Number of human cases": "human_cases",
        "Number of confirmed human cases": "confirmed_human_cases"
    },
}

def create_resilient_session():
    session = requests.Session()
    retries = Retry(
        total=3,
        backoff_factor=3,
        status_forcelist=[429, 500, 502, 503, 504],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retries)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def load_file_safely(session, year, url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    response = session.get(url, headers=headers)
    response.raise_for_status()

    if ".csv" in url:
        csv_data = StringIO(response.content.decode("latin1"))
        return pd.read_csv(csv_data, sep=None, engine="python")
    elif ".xlsx" in url or ".xls" in url:
        excel_data = BytesIO(response.content)
        if year == 2020:
            return pd.read_excel(excel_data, sheet_name="2020 data (01022021)")
        elif year in (2021, 2023):
            # Read first available sheet safely if named sheet varies
            xl = pd.ExcelFile(excel_data)
            sheet = "OverviewTable" if "OverviewTable" in xl.sheet_names else 0
            return pd.read_excel(xl, sheet_name=sheet)
        else:
            return pd.read_excel(excel_data)

def parse_excel_date(val):
    if pd.isna(val):
        return pd.NaT
    try:
        val_float = float(val)
        return pd.to_datetime(val_float, unit="D", origin="1899-12-30")
    except (ValueError, TypeError):
        pass
    return pd.to_datetime(val, errors="coerce")

def get_country_info(value):
    value = str(value).strip()

    # Case 1: ISO code -> get country name
    if len(value) == 2:
        country = pycountry.countries.get(alpha_2=value.upper())
        if country:
            return pd.Series([country.name, country.alpha_2])

    # Case 2: Country name -> get ISO code
    else:
        try:
            country = pycountry.countries.search_fuzzy(value)[0]
            return pd.Series([country.name, country.alpha_2])
        except LookupError:
            pass

    return pd.Series([None, None])

In [ ]:
session = create_resilient_session()
dfs_all = []

for year, url in file_urls.items():
    try:
        df = load_file_safely(session, year, url)

        # Normalize headers
        df.columns = [" ".join(str(col).split()) for col in df.columns]
        mapping = column_mapping.get(year, {})
        df = df.rename(columns=mapping)

        # Retain target mapped columns
        target_cols = list(set(mapping.values()))
        df = df[[c for c in target_cols if c in df.columns]].copy()

        # Extract or fill country 
        if "country" in df.columns:
            df["country"] = df["country"].ffill()
        elif "location_code" in df.columns:
            # First 2 letters of NUTS code give country code (e.g. IT, EL, FR)
            df["country"] = df["location_code"].astype(str).str[:2].str.upper()

        # Dynamic cleanup filter based ONLY on existing columns
        filter_cols = [c for c in ["country", "location_name", "location_code"] if c in df.columns]
        for col in filter_cols:
            df = df[
                ~df[col]
                .astype(str)
                .str.contains("total|sum|eu/eea|unscr|designation", case=False, na=False)
            ]

        if filter_cols:
            df = df.dropna(how="all", subset=filter_cols)

        df["year"] = year

        if "first_case_date" in df.columns:
            df["first_case_date"] = df["first_case_date"].apply(parse_excel_date)

        dfs_all.append(df)
        print(f"Successfully processed year {year}: {len(df)} rows")
        time.sleep(2)

    except Exception as e:
        print(f"Error processing year {year}: {e}")

if dfs_all:
    df_ecdc = pd.concat(dfs_all, ignore_index=True)

    numeric_cols = ["human_cases", "confirmed_human_cases"]
    for col in numeric_cols:
        if col in df_ecdc.columns:
            df_ecdc[col] = (
                pd.to_numeric(df_ecdc[col], errors="coerce")
                .fillna(0)
                .astype(int)
            )

    if "human_cases" in df_ecdc.columns:
        df_ecdc = df_ecdc[df_ecdc["human_cases"] != 0]

    # Filter by target countries using both names and codes since ECDC data are heterogeneous across years
    target_countries = ["Italy", "Greece", "Germany", "Spain", "Hungary", "IT","GR", "DE","ES"]
    df_ecdc = df_ecdc[df_ecdc["country"].isin(target_countries)]

    # Normalize country name
    df_ecdc[["country_name", "country_iso"]] = df_ecdc["country"].apply(get_country_info)

    final_cols = [
        "year",
        "country_name",
        "country_iso",
        "location_name",
        "first_case_date",
        "human_cases",
        "confirmed_human_cases"
    ]
    final_cols = [c for c in final_cols if c in df_ecdc.columns]
    df_ecdc = df_ecdc[final_cols]

    # Sort by country and date
    df_ecdc.to_excel("ecdc_wn_2018_2023.xlsx", index=False)

In [ ]:
df_ecdc

# DOWNLOAD ITALIAN DATA

In [ ]:
dfs_all = []
for year in range(2018,2026):
  df = pd.read_csv(f"https://raw.githubusercontent.com/fbranda/west-nile/main/{year}/national-trend/wn-ita-national-trend-{year}.csv")
  dfs_all.append(df)

if dfs_all:
    df_it = pd.concat(dfs_all, ignore_index=True)

In [ ]:
# Filter by humans
df_it_hum = df_it.loc[df_it["host"]=="humans"][["data","new_cases","total_cases"]]
df_it_hum.to_excel("italy_wn_2018_2025.xlsx", index = False)